# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
-- Change to your user's schema
USE SCHEMA data5035.GORILLA;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata limit 10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [ ]:
USE ROLE GORILLA_DATA5035_ROLE;
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    import math

    dx = x - origin_x
    dy = y - origin_y

    tile_width = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y

    tile_col = int(dx // tile_width)
    tile_row = int(dy // tile_height)

    tile_col_letter = chr(ord('A') + tile_col)
    tile_row_letter = chr(ord('A') + tile_row)
    tile = tile_row_letter + tile_col_letter

    local_x = dx - tile_col * tile_width
    local_y = dy - tile_row * tile_height

    su_col = int(local_x // su_size)
    su_row = int(local_y // su_size)

    su_row_from_top = (tile_grid_y - 1) - su_row
    su = int(su_row_from_top * tile_grid_x + su_col + 1)

    subcell_size = su_size / subcell_grid
    sc_local_x = local_x - su_col * su_size
    sc_local_y = local_y - su_row * su_size

    sc_col = int(sc_local_x // subcell_size)
    sc_row = int(sc_local_y // subcell_size)

    sc = int(sc_row * subcell_grid + sc_col + 1)

    return {'tile': tile, 'survey_unit': su, 'subcell': sc}
    $$;

In [ ]:
%%sql -r dataframe_3
    CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [ ]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Compute Layered Averages

In [ ]:
%%sql -r dataframe_8

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [ ]:
%%sql -r dataframe_9

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [ ]:
%%sql -r dataframe_10

In [ ]:
CREATE OR REPLACE FUNCTION CONVERT_XY(
    X FLOAT,
    Y FLOAT,
    ORIGIN_X FLOAT,
    ORIGIN_Y FLOAT,
    SU_SIZE FLOAT,
    TILE_GRID_X NUMBER(38,0),
    TILE_GRID_Y NUMBER(38,0),
    SUBCELL_GRID NUMBER(38,0)
)
RETURNS OBJECT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'convert_xy'
AS 
$$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    dx = x - origin_x
    dy = y - origin_y

    tile_width = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y

    # --- TILE INDEX ---
    tile_col = int(dx // tile_width)
    tile_row = int(dy // tile_height)

    # Column letter first (matches 'AA', 'AB', etc.)
    tile = chr(ord('A') + tile_col) + chr(ord('A') + tile_row)

    # --- LOCAL POSITION INSIDE TILE ---
    local_x = dx - tile_col * tile_width
    local_y = dy - tile_row * tile_height

    # --- SURVEY UNIT INDEX ---
    su_col = int(local_x // su_size)
    su_row = int(local_y // su_size)

    # Flip Y so origin is top-left (matches your examples)
    su_row_from_top = (tile_grid_y - 1) - su_row

    # IMPORTANT: numbering appears column-major in your expected output
    su = int(su_col * tile_grid_y + su_row_from_top + 1)

    # --- SUBCELL INDEX ---
    subcell_size = su_size / subcell_grid

    sc_local_x = local_x - su_col * su_size
    sc_local_y = local_y - su_row * su_size

    sc_col = int(sc_local_x // subcell_size)
    sc_row = int(sc_local_y // subcell_size)

    # Subcell numbering (row-major is typical; adjust if needed)
    sc = int(sc_row * subcell_grid + sc_col + 1)

    return {
        'tile': tile,
        'survey_unit': su,
        'subcell': sc
    }
$$;

In [ ]:
SELECT CONVERT_XY(
    2180160.0001,
    6660000.0000,
    <origin_x>,
    <origin_y>,
    <su_size>,
    <tile_grid_x>,
    <tile_grid_y>,
    <subcell_grid>
);